# Pretrained Model Demo: Geometric Markers from ImageNet Features

This notebook illustrates the core Section 4 analysis from the paper:

> **Diagnosing Generalization Failures from Representational Geometry Markers**  
> Chi-Ning Chou, Artem Kirsanov, Yao-Yuan Yang, SueYeon Chung — *ICLR 2026*

We compare **ResNet-50 V1** (older training recipe) vs **ResNet-50 V2** (improved recipe) by computing
representational geometry markers—effective dimension D<sub>eff</sub>, radius R<sub>eff</sub>, and
utility Ψ<sub>eff</sub>—from their penultimate-layer (avgpool) features on a small ImageNet subset.

Following the paper's protocol, we sample **2 random ImageNet classes × 50 images** and compute
GLUE geometry markers on the resulting feature manifolds.

**What you need:**
- ImageNet validation set arranged as `val_root/{synset_id}/*.JPEG`  
  (set `IMAGENET_VAL_ROOT` in the Config cell below)
- The `analysis` package and `glue` package (included in `environment.yml`)

**Expected runtime:** ~2 min on CPU, ~30 s on GPU.

**Full reproduction of Fig 5** requires all 20 architectures × 9 OOD transfer datasets;
see `figures.ipynb` and the paper appendix for details.

In [1]:
import os
import sys
import random

import numpy as np
import pandas as pd
import torch
import torchvision.models as tvm
from torchvision.datasets import ImageFolder

sys.path.insert(0, os.path.abspath(''))
from analysis import geo_analysis

In [2]:
# ── Config ────────────────────────────────────────────────────────────────────
IMAGENET_VAL_ROOT = '/path/to/imagenet/val'  # adjust to your local path

NUM_CLASSES = 2    # paper protocol: 2 random ImageNet classes
M           = 50   # samples per class (ImageNet val has exactly 50/class)
SEED        = 42

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cpu


In [3]:
# ── Helpers ───────────────────────────────────────────────────────────────────

class SubsetImageFolder(ImageFolder):
    """ImageFolder restricted to a specific list of class directories."""
    def __init__(self, root, transform, selected_classes):
        self._selected = sorted(selected_classes)
        super().__init__(root, transform=transform)

    def find_classes(self, directory):
        class_to_idx = {c: i for i, c in enumerate(self._selected)}
        return self._selected, class_to_idx


def extract_avgpool_features(model, dataloader, device):
    """Run forward passes and collect avgpool features, sorted by class label."""
    buf = []
    handle = model.avgpool.register_forward_hook(
        lambda m, inp, out: buf.append(torch.flatten(out, 1).detach().cpu())
    )
    feat_list, label_list = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in dataloader:
            model(images.to(device))
            feat_list.append(buf.pop())
            label_list.append(labels)
    handle.remove()
    features = torch.cat(feat_list).numpy()   # (N, D)
    labels   = torch.cat(label_list).numpy()  # (N,)
    sort_idx = np.argsort(labels, kind='stable')
    return features[sort_idx]  # class-ordered: M rows per class

In [4]:
# ── Sample ImageNet classes ───────────────────────────────────────────────────
all_synsets = sorted(os.listdir(IMAGENET_VAL_ROOT))
rng = random.Random(SEED)
selected_classes = sorted(rng.sample(all_synsets, NUM_CLASSES))
print(f'Selected {NUM_CLASSES} synsets: {selected_classes}')

Selected 2 synsets: ['n01945685', 'n03769881']


In [5]:
# ── Extract features and compute markers for V1 and V2 ───────────────────────

WEIGHT_CONFIGS = {
    'V1': tvm.ResNet50_Weights.IMAGENET1K_V1,
    'V2': tvm.ResNet50_Weights.IMAGENET1K_V2,
}

results = {}

for weight_name, weights in WEIGHT_CONFIGS.items():
    print(f'\n── ResNet-50 {weight_name} ──────────────────────')

    # Load model with official pretrained weights
    model = tvm.resnet50(weights=weights).to(DEVICE)
    model.eval()

    # Build dataloader using the model's recommended preprocessing
    transform = weights.transforms()
    dataset   = SubsetImageFolder(IMAGENET_VAL_ROOT, transform, selected_classes)
    loader    = torch.utils.data.DataLoader(
        dataset, batch_size=64, shuffle=False, num_workers=0, pin_memory=False
    )
    print(f'  Dataset: {len(dataset)} images ({NUM_CLASSES} classes × {M} samples)')

    # Extract avgpool features (2048-dim for ResNet-50)
    X = extract_avgpool_features(model, loader, DEVICE)  # (NUM_CLASSES*M, 2048)
    print(f'  Feature matrix: {X.shape}')

    # Compute geometric markers via GLUE
    geo = geo_analysis(X, num_classes=NUM_CLASSES, M=M)
    results[weight_name] = geo
    print(f'  dimension={geo["dimension"]:.3f}  radius={geo["radius"]:.3f}  '
          f'utility={geo["utility"]:.3f}  PR={geo["participation_ratio"]:.3f}')


── ResNet-50 V1 ──────────────────────


  Dataset: 100 images (2 classes × 50 samples)


  Feature matrix: (100, 2048)


  dimension=10.027  radius=0.754  utility=0.901  PR=8.738

── ResNet-50 V2 ──────────────────────


  Dataset: 100 images (2 classes × 50 samples)


  Feature matrix: (100, 2048)


  dimension=9.415  radius=0.842  utility=0.946  PR=9.774


In [6]:
# ── Summary table ─────────────────────────────────────────────────────────────

marker_labels = {
    'dimension':           'D_eff (manifold dimension)',
    'radius':              'R_eff (manifold radius)',
    'utility':             'Ψ_eff (manifold utility)',
    'participation_ratio': 'PR (participation ratio)',
    'neural_collapse':     'NC1 (neural collapse)',
}

rows = []
for key, label in marker_labels.items():
    v1 = results['V1'][key]
    v2 = results['V2'][key]
    rows.append({'Marker': label, 'V1': v1, 'V2': v2, 'Δ (V2 − V1)': v2 - v1})

df_cmp = pd.DataFrame(rows).set_index('Marker')
print(df_cmp.to_string(float_format='{:.4f}'.format))

                                V1     V2  Δ (V2 − V1)
Marker                                                
D_eff (manifold dimension) 10.0270 9.4155      -0.6115
R_eff (manifold radius)     0.7539 0.8418       0.0878
Ψ_eff (manifold utility)    0.9011 0.9456       0.0445
PR (participation ratio)    8.7377 9.7739       1.0362
NC1 (neural collapse)       0.0310 0.0133      -0.0178


## Interpretation

The paper (Fig 5) shows that across 20 pretrained architectures, models where **V2 > V1** on
ImageNet top-1 accuracy tend to also have **higher** manifold utility Ψ<sub>eff</sub> and **lower**
effective dimension D<sub>eff</sub> — a geometry consistent with better OOD transfer.

Key takeaways:
- **Higher Ψ<sub>eff</sub> (utility)** → manifolds are well-separated and linearly decodable;
  this correlates with better OOD linear-probe accuracy across 9 downstream datasets.
- **Lower D<sub>eff</sub> (dimension)** → more compact, less diffuse representations;
  also correlated with OOD transfer.
- **R<sub>eff</sub> (radius)** has a weaker and sometimes opposite effect depending on the dataset.

For the full 20-model comparison and all 9 OOD datasets, see `figures.ipynb` and the pre-saved
results in `data/df_fig15-20.pkl`.